# Binary Classification using Deep Neural Network
## Classify Movie Reviews as Positive or Negative (IMDB Dataset)

## Step 1: Import Libraries

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

## Step 2: Load the IMDB Dataset
Keras has the IMDB dataset built-in. We keep only the top 10,000 most common words.

In [ ]:
# Load dataset - only use top 10,000 words
NUM_WORDS = 10000

(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=NUM_WORDS)

print("Training samples:", len(x_train))
print("Testing samples:", len(x_test))
print("\nExample label (0=negative, 1=positive):", y_train[0])
print("Example review (as word indices):", x_train[0][:10], "...")

## Step 3: Preprocess Data
Each review has different length. We convert each review into a fixed-size vector of 0s and 1s (multi-hot encoding).
- If word 5 appears in the review → position 5 = 1
- Otherwise → position 5 = 0

In [ ]:
def vectorize_sequences(sequences, dimension=NUM_WORDS):
    # Create a matrix of zeros
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0  # Set positions of words to 1
    return results

# Vectorize the data
x_train_vec = vectorize_sequences(x_train)
x_test_vec  = vectorize_sequences(x_test)

# Convert labels to float
y_train = y_train.astype('float32')
y_test  = y_test.astype('float32')

print("Shape of x_train_vec:", x_train_vec.shape)  # (25000, 10000)
print("Shape of x_test_vec: ", x_test_vec.shape)   # (25000, 10000)

## Step 4: Build the Deep Neural Network
Our model:
- **Input**: 10,000-dimensional vector
- **Hidden Layer 1**: 16 neurons, ReLU activation
- **Hidden Layer 2**: 16 neurons, ReLU activation
- **Output Layer**: 1 neuron, Sigmoid activation (gives probability between 0 and 1)

In [ ]:
model = keras.Sequential([
    layers.Dense(16, activation='relu', input_shape=(NUM_WORDS,)),  # Hidden Layer 1
    layers.Dense(16, activation='relu'),                             # Hidden Layer 2
    layers.Dense(1,  activation='sigmoid')                          # Output Layer
])

model.summary()

## Step 5: Compile the Model
- **Loss**: binary_crossentropy (used for binary classification)
- **Optimizer**: adam
- **Metric**: accuracy

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## Step 6: Train the Model
We use a validation split to monitor performance during training.

In [ ]:
history = model.fit(
    x_train_vec, y_train,
    epochs=10,
    batch_size=512,
    validation_split=0.2,   # Use 20% of training data for validation
    verbose=1
)

## Step 7: Plot Training Results

In [ ]:
# Plot Accuracy
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

## Step 8: Evaluate on Test Data

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test_vec, y_test, verbose=0)

print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

## Step 9: Predict on New Reviews
Let's see what the model predicts for a few test samples.

In [ ]:
# Predict on first 5 test samples
predictions = model.predict(x_test_vec[:5])

print("Sample Predictions:")
print("-" * 40)
for i in range(5):
    prob = predictions[i][0]
    sentiment = "Positive 😊" if prob >= 0.5 else "Negative 😞"
    actual    = "Positive 😊" if y_test[i] == 1 else "Negative 😞"
    print(f"Review {i+1}: Predicted={sentiment} (prob={prob:.2f}), Actual={actual}")

## Summary

| Step | What We Did |
|------|-------------|
| 1 | Imported TensorFlow & Keras |
| 2 | Loaded IMDB dataset (25,000 train + 25,000 test reviews) |
| 3 | Converted reviews to vectors (multi-hot encoding) |
| 4 | Built DNN: Input → Dense(16) → Dense(16) → Dense(1, sigmoid) |
| 5 | Compiled with binary_crossentropy loss & adam optimizer |
| 6 | Trained for 10 epochs |
| 7 | Plotted accuracy and loss curves |
| 8 | Evaluated on test set (~87-88% accuracy) |
| 9 | Made predictions on new samples |

**Result**: The model achieves ~87% accuracy on classifying movie reviews as positive or negative!